# Bias testing on `data/sample_loan_data.csv`

This notebook audits a loan-approval model for gender bias using the repository's sample fixture.

Flow:

1. Set up reproducible imports and thresholds.
2. Profile the fixture and measure bias already present in historical labels.
3. Train a model that excludes `gender` from its features.
4. Audit predictions with `bias_detection.run_bias_audit`.
5. Explicitly check equalized odds, which the helper computes but does not enforce.
6. Run an optional mitigation experiment fitted on training data only.
7. Summarize limitations.

Thresholds are illustrative policy choices, not legal conclusions. The notebook does not write files.


## 1. Setup

Complete Phase A first so `bias_detection` resolves to the repo `src/` tree:

```
python -m pip install fairlearn scikit-learn pandas numpy huggingface_hub pytest pytest-cov -e ./src/model_card_generator -e ./src/bias_detection -e ./src/audit_trail
```


In [ ]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import sklearn
import fairlearn
import bias_detection
from bias_detection.bias_detection import print_bias_report, run_bias_audit
from fairlearn.metrics import equalized_odds_difference
from IPython.display import display

SEED = 42
DP_THRESHOLD = 0.10
DI_THRESHOLD = 0.80
EO_THRESHOLD = 0.15


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "sample_loan_data.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find data/sample_loan_data.csv above " + str(start)
    )


repo_root = find_repo_root(Path.cwd().resolve())
data_path = repo_root / "data" / "sample_loan_data.csv"

assert Path(bias_detection.__file__).resolve().is_relative_to(repo_root / "src"), (
    "bias_detection must resolve to the repo src/ tree; reinstall with "
    "python -m pip install -e ./src/bias_detection"
)

print(f"repo_root: {repo_root}")
print(f"data_path: {data_path}")
print(f"bias_detection: {bias_detection.__file__}")
print(
    f"python: {platform.python_version()}, pandas: {pd.__version__}, "
    f"numpy: {np.__version__}, scikit-learn: {sklearn.__version__}, "
    f"fairlearn: {fairlearn.__version__}"
)

## 2. Load and profile the dataset

The dataset has 200 rows, five columns, and no missing values. `approved` is the binary target; `gender` is the protected attribute used for auditing.


In [ ]:
df = pd.read_csv(data_path)
print(f"rows={len(df)}, columns={list(df.columns)}")
display(df.head())

print("\nMissing values:")
display(df.isna().sum().to_frame("missing"))

print("\nNumeric summary:")
display(df.describe())

## 3. Historical bias in the labels

Before training anything, measure selection rates in the historical decisions. These labels may already encode biased outcomes, so they are a baseline, not ground truth.


In [ ]:
group_rates = (
    df.groupby("gender")["approved"]
    .mean()
    .rename("selection_rate")
    .to_frame()
)
display(group_rates)

hist_dp = group_rates["selection_rate"].max() - group_rates["selection_rate"].min()
hist_di = group_rates["selection_rate"].min() / group_rates["selection_rate"].max()

print(f"Historical demographic parity difference: {hist_dp:.4f} (threshold {DP_THRESHOLD})")
print(f"Historical disparate impact ratio:       {hist_di:.4f} (threshold {DI_THRESHOLD})")
print(f"Historical DP violation: {hist_dp > DP_THRESHOLD}")
print(f"Historical DI violation: {hist_di < DI_THRESHOLD}")

## 4. Train a model without `gender` as a feature

`gender` is withheld from training but retained for the fairness audit. This demonstrates that excluding a protected attribute does not by itself eliminate disparities.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

features = ["income", "credit_score", "debt_ratio"]
X = df[features]
y = df["approved"]
sensitive = df["gender"]

X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive, test_size=0.30, random_state=SEED, stratify=y
)

model = GradientBoostingClassifier(n_estimators=100, random_state=SEED)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"train rows={len(X_train)}, test rows={len(X_test)}")
print(f"accuracy={accuracy_score(y_test, y_pred):.4f}")

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual 0", "actual 1"],
    columns=["pred 0", "pred 1"],
)
display(cm)

## 5. Audit model predictions

`run_bias_audit` checks demographic parity and disparate impact. It computes `equalized_odds_diff` but does not include it in `violations` or `passed`, so we check that separately in the next section.


In [ ]:
audit = run_bias_audit(
    y_true=y_test.to_numpy(),
    y_pred=y_pred,
    sensitive_features=sens_test,
    demographic_parity_threshold=DP_THRESHOLD,
    disparate_impact_threshold=DI_THRESHOLD,
)

print_bias_report(audit)
display(audit["metric_frame"].by_group)
print("library passed:", audit["passed"])
print("library violations:", audit["violations"])

## 6. Close the gap: equalized odds

Check `equalized_odds_difference` explicitly and compare it with the helper's `passed` flag.


In [ ]:
eo_diff = equalized_odds_difference(
    y_true=y_test.to_numpy(),
    y_pred=y_pred,
    sensitive_features=sens_test,
)

print(f"Equalized odds difference: {eo_diff:.4f} (threshold {EO_THRESHOLD})")
print(f"EO violation: {eo_diff > EO_THRESHOLD}")
print(f"Note: audit['passed'] is {audit['passed']}, so the helper summary hides this EO result.")

## 7. Optional mitigation experiment

Fit a demographic-parity mitigator on the **training** data only, then re-audit the same held-out test set. This avoids the leakage pattern in `examples/example_bias_audit.py`, which fits `ThresholdOptimizer` on test data. On this small fixture the mitigator may worsen some metrics; treat the comparison as illustrative, not a recommendation.


In [ ]:
from fairlearn.reductions import DemographicParity, ExponentiatedGradient

mitigator = ExponentiatedGradient(
    estimator=GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    constraints=DemographicParity(),
)
mitigator.fit(X_train, y_train, sensitive_features=sens_train)
y_pred_mitigated = mitigator.predict(X_test)

mitigated_audit = run_bias_audit(
    y_true=y_test.to_numpy(),
    y_pred=y_pred_mitigated,
    sensitive_features=sens_test,
    demographic_parity_threshold=DP_THRESHOLD,
    disparate_impact_threshold=DI_THRESHOLD,
)
mitigated_eo = equalized_odds_difference(
    y_true=y_test.to_numpy(),
    y_pred=y_pred_mitigated,
    sensitive_features=sens_test,
)

comparison = pd.DataFrame(
    [
        {
            "model": "baseline",
            "accuracy": accuracy_score(y_test, y_pred),
            "dp_diff": audit["demographic_parity_diff"],
            "di_ratio": audit["disparate_impact_ratio"],
            "eo_diff": eo_diff,
            "library_passed": audit["passed"],
        },
        {
            "model": "mitigated",
            "accuracy": accuracy_score(y_test, y_pred_mitigated),
            "dp_diff": mitigated_audit["demographic_parity_diff"],
            "di_ratio": mitigated_audit["disparate_impact_ratio"],
            "eo_diff": mitigated_eo,
            "library_passed": mitigated_audit["passed"],
        },
    ]
).set_index("model")

display(comparison)

## 8. Interpretation and limitations

- Historical labels show a large gender approval-rate gap; they are a biased baseline, not neutral ground truth.
- Dropping `gender` from the model features does not guarantee fair outcomes; other features can act as proxies.
- `audit["passed"] == True` only means demographic parity and disparate impact passed. Always check equalized odds separately until `run_bias_audit` enforces it.
- The fixture has 200 rows and this split has 60 test rows, so group estimates are noisy.
- Only binary `gender` is audited; no intersectional or other protected attributes are analyzed.
- Thresholds here are illustrative policy choices, not legal or regulatory advice.
- The notebook is read-only with respect to the repository.
